# Unit execution for `algo/dilation_algo.py`

This notebook checks the two core algorithmic pieces from Schlimgen et al. (2022):
1. diagonal one-ancilla dilation for probabilistic state preparation,
2. SVD-based one-ancilla dilation for a general nonunitary operator.

In [1]:
import os
import sys
import numpy as np

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from algo.dilation_algo import (
    build_state_preparation_dilation,
    prepare_subnormalized_state_from_uniform_superposition,
    build_one_ancilla_svd_dilation,
    apply_one_ancilla_dilation_to_state,
)

## 1. Probabilistic preparation of a subnormalised one-qubit state

In [2]:
target_amplitudes = np.array([0.61 + 0.10j, 0.35 - 0.20j], dtype=complex)
diag_dilation = build_state_preparation_dilation(target_amplitudes)
prep = prepare_subnormalized_state_from_uniform_superposition(target_amplitudes)

print('Sigma entries           =', diag_dilation.diagonal)
print('Sigma_plus entries      =', diag_dilation.sigma_plus)
print('Sigma_minus entries     =', diag_dilation.sigma_minus)
print('Success probability     =', prep.p_success)
print('Failure probability     =', prep.p_failure)
print('State error norm        =', prep.state_error_norm)
print('Ancilla-|0> branch      =', prep.ancilla_zero_branch)
print('Target state            =', prep.target_scaled_state)
print('Allclose?               =', np.allclose(prep.ancilla_zero_branch, prep.target_scaled_state))

Sigma entries           = [0.61+0.1j 0.35-0.2j]
Sigma_plus entries      = [0.48283413+0.87571183j 0.80404168+0.59457294j]
Sigma_minus entries     = [ 0.73716587-0.67571183j -0.10404168-0.99457294j]
Success probability     = 0.2722999999999999
Failure probability     = 0.7276999999999998
State error norm        = 1.2098374922832816e-16
Ancilla-|0> branch      = [0.43133514+0.07071068j 0.24748737-0.14142136j]
Target state            = [0.43133514+0.07071068j 0.24748737-0.14142136j]
Allclose?               = True


## 2. SVD-based one-ancilla dilation for a general nonunitary operator

In [4]:
M = np.array([
    [0.56, 0.15 - 0.05j],
    [0.10 + 0.07j, 0.55*np.exp(1j*0.30)]
], dtype=complex)

psi = np.array([1.0, 1.0j], dtype=complex)
psi = psi / np.linalg.norm(psi)

dilation = build_one_ancilla_svd_dilation(M, auto_scale=True)
sim = apply_one_ancilla_dilation_to_state(M, psi, auto_scale=True)

print('Scale factor                 =', dilation.scale_factor)
print('Singular values              =', dilation.singular_values)
print('Unitarity check              =', np.allclose(dilation.full_unitary.conj().T @ dilation.full_unitary, np.eye(2*M.shape[0])))
print('Postselected ancilla-|0>     =', sim.ancilla_zero_branch)
print('Target scaled state          =', sim.target_scaled_state)
print('State error norm            =', sim.state_error_norm)
print('Success probability         =', sim.p_success)
print('Failure probability         =', sim.p_failure)
print('Allclose?                   =', np.allclose(sim.ancilla_zero_branch, sim.target_scaled_state))

Scale factor                 = 1.0
Singular values              = [0.69546985 0.41511647]
Unitarity check              = True
Postselected ancilla-|0>     = [ 0.43133514+0.10606602j -0.04421971+0.42103618j]
Target scaled state          = [ 0.43133514+0.10606602j -0.04421971+0.42103618j]
State error norm            = 2.5514002453611344e-16
Success probability         = 0.3765268434649618
Failure probability         = 0.6234731565350373
Allclose?                   = True
